# TokaGym Closed-Loop Policy Evaluation Check

Interactive verification check for TokaGym TCV closed-loop plasma dynamics and rampdown control using **BAUX**.

In [1]:
# 1. Environment & Setup
import os
import sys
from pathlib import Path

os.environ.setdefault("JAX_PRECISION", "f32")

root = Path.cwd()
while root.name and not (root / "python" / "baux").exists():
    root = root.parent
if str(root / "python") not in sys.path:
    sys.path.insert(0, str(root / "python"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import numpy as np
from evals.tokagym.tokagym_rollout import (
    ACTION_DIM,
    OBS_DIM,
    TOTAL_PARAMS,
    kaiming_bounds,
)

print(f"TokaGym Policy Spec: {OBS_DIM} inputs -> 256x256x256 -> {ACTION_DIM} outputs")
print(f"Total parameter vector size: {TOTAL_PARAMS:,} weights")

TokaGym Policy Spec: 11 inputs -> 256x256x256 -> 8 outputs
Total parameter vector size: 136,712 weights


## 2. Policy Initialization & Parameter Bounds Check

Verify layer-wise Kaiming parameter distributions and actuator saturation bounds.

In [2]:
# 2. Parameter Distribution Inspection
bounds = kaiming_bounds()
print(f"Parameter bounds shape: {bounds.shape}")
print(f"Weight interval range: [{bounds[:, 0].min():.4f}, {bounds[:, 1].max():.4f}]")

# Sample an initial candidate
rng = np.random.default_rng(42)
theta_0 = rng.uniform(bounds[:, 0], bounds[:, 1])
print(f"Sampled theta_0: L2 norm = {np.linalg.norm(theta_0):.2f}, mean = {theta_0.mean():.5f}")

Parameter bounds shape: (136712, 2)
Weight interval range: [-0.5222, 0.5222]
Sampled theta_0: L2 norm = 28.03, mean = 0.00009


## 3. High-Dimensional Plasma Metrics & Reward Combiner

Evaluate soft and hard physics operational limits (Greenwald density fraction $f_{\text{GW}}$, safety factor $q_{95} / \iota_{95}$, poloidal beta $\beta_p$, and elongation/minor-radius aspect ratio).

In [3]:
# 3. Tokamak Constraint Envelope
metrics = [
    ("Greenwald Fraction (f_GW)", 0.85, 1.05),
    ("Safety Factor (iota_95)", 0.25, 0.20),
    ("Poloidal Beta (beta_p)", 1.20, 1.80),
    ("Elongation Ratio (a / kappa)", 0.22, 0.35),
]

names, soft_lims, hard_lims = zip(*metrics)
x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4), dpi=150)
ax.bar(x - width/2, soft_lims, width, label="Soft Operational Limit", color="#ff7f0e", alpha=0.85)
ax.bar(x + width/2, hard_lims, width, label="Hard Disruption Boundary", color="#d62728", alpha=0.85)

ax.set_ylabel("Normalized Physics Units")
ax.set_title("TCV Tokamak Plasma Operational Envelope (TokaGym)")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15, ha="right")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()